# AI Movie Studio — Google Colab
Custom HTML/CSS/JS frontend + FastAPI backend. Colab runs the workspace; generation is delegated to configured provider APIs. Mount Google Drive to keep project files between sessions.

In [ ]:
import os, pathlib
repo_dir = pathlib.Path('/content/ai-movie-studio')
if not repo_dir.exists():
    !git clone --depth 1 https://github.com/PandaCwel/Ai-video-maker.git /content/ai-movie-studio
%cd /content/ai-movie-studio


In [ ]:
!apt-get -qq update && apt-get -qq install -y ffmpeg
!pip -q install -r requirements.txt


In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')
os.environ['STUDIO_DATA_DIR'] = '/content/drive/MyDrive/AI-Movie-Studio/data'
for secret in ['OPENAI_API_KEY','OPENAI_BASE_URL','OPENAI_MODEL','REPLICATE_API_TOKEN','REPLICATE_IMAGE_MODEL','REPLICATE_VIDEO_MODEL','REPLICATE_MUSIC_MODEL','REPLICATE_LIPSYNC_MODEL','ELEVENLABS_API_KEY']:
    try:
        value = userdata.get(secret)
        if value: os.environ[secret] = value
    except Exception:
        pass
print('Drive persistence configured. Provider secrets loaded when available.')


In [ ]:
import subprocess, sys, time
server = subprocess.Popen([sys.executable, '-m', 'uvicorn', 'app.main:app', '--host', '0.0.0.0', '--port', '8000'])
time.sleep(4)
from google.colab.output import eval_js
studio_url = eval_js('google.colab.kernel.proxyPort(8000)')
print('Open AI Movie Studio:', studio_url)


## Configure provider secrets
In Colab open **Secrets** (key icon) and add only the services you use. Required keys: `OPENAI_API_KEY` for screenplay analysis and shot planning; `REPLICATE_API_TOKEN` plus the model IDs for image/video/music/lip-sync; `ELEVENLABS_API_KEY` for speech and consent-based voice cloning. Model IDs and provider input schemas vary, so choose models compatible with the adapter in `app/services/production.py`. Never publish your keys or the Colab proxy URL.

## Important limits
This notebook does not turn Colab Free into a guaranteed 30-minute video GPU. Long-form production needs multiple generated shots, paid/provider quotas or an available GPU, storage and rendering time. The current editor concatenates generated/uploaded clips; it is not yet a frame-accurate nonlinear editor. Keep the runtime active during jobs.